# AI Agents with LangGraph Part 1

In [ ]:
%pip install -qU langgraph langchain-google-genai

## Exercise 1: SQL Agents

SQL agents adalah AI Agent yang dapat mengambil data dari table SQL dan menganalisis sesuai dengan pertanyaan dari user.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os

from google.colab import userdata
GEMINI = userdata.get('GEMINI')
os.environ["GOOGLE_API_KEY"] = GEMINI

### Langkah 1: Define beberapa table SQL

In [ ]:
%load_ext sql
%sql sqlite:///sample.db

In [ ]:
%%sql
-- Create the 'products' table
CREATE TABLE IF NOT EXISTS products (
    product_id   INTEGER PRIMARY KEY AUTOINCREMENT,
    product_name VARCHAR(255) NOT NULL,
    price        DECIMAL(10, 2) NOT NULL
);

-- Create the 'staff' table
CREATE TABLE IF NOT EXISTS staff (
    staff_id   INTEGER PRIMARY KEY AUTOINCREMENT,
    first_name VARCHAR(255) NOT NULL,
    last_name  VARCHAR(255) NOT NULL
);

-- Create the 'orders' table
CREATE TABLE IF NOT EXISTS orders (
    order_id      INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_name VARCHAR(255) NOT NULL,
    staff_id      INTEGER NOT NULL,
    product_id    INTEGER NOT NULL,
    FOREIGN KEY (staff_id)   REFERENCES staff (staff_id),
    FOREIGN KEY (product_id) REFERENCES products (product_id)
);

-- Insert data into the 'products' table
INSERT INTO products (product_name, price) VALUES
    ('Laptop', 799.99), ('Keyboard', 129.99), ('Mouse', 29.99);

-- Insert data into the 'staff' table
INSERT INTO staff (first_name, last_name) VALUES
    ('Alice', 'Smith'), ('Bob', 'Johnson'), ('Charlie', 'Williams');

-- Insert data into the 'orders' table
INSERT INTO orders (customer_name, staff_id, product_id) VALUES
    ('David Lee', 1, 1), ('Emily Chen', 2, 2), ('Frank Brown', 1, 3);

In [ ]:
import sqlite3

db_file = "sample.db"

### Langkah 2: Define Tools

Tools yang akan digunakan pada SQL agent adalah `list_tables`, `describe_tables`, dan `execute_query`
sehingga AI dapat memahami metadata table terlebih dahulu sebelum mulai menjalankan query

In [ ]:
def list_tables() -> list[str]:
    """Retrieve the names of all tables in the database."""
    print(' - DB CALL: list_tables')
    with sqlite3.connect(db_file) as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
        tables = cursor.fetchall()
        return [t[0] for t in tables]

list_tables()

In [ ]:
def describe_table(table_name: str) -> list[tuple[str, str]]:
    """Look up the table schema.

    Returns:
      List of columns, where each entry is a tuple of (column, type).
    """
    print(' - DB CALL: describe_table')
    with sqlite3.connect(db_file) as conn:
        cursor = conn.cursor()
        cursor.execute(f"PRAGMA table_info({table_name});")
        schema = cursor.fetchall()
        return [(col[1], col[2]) for col in schema]

describe_table("products")

In [ ]:
def execute_query(sql: str) -> list[list[str]]:
    """Execute a SELECT statement, returning the results."""
    print(' - DB CALL: execute_query')
    with sqlite3.connect(db_file) as conn:
        cursor = conn.cursor()
        cursor.execute(sql)
        return cursor.fetchall()

execute_query("select * from products")

### Langkah 3: Bangun chatbot dengan create_react_agent

In [ ]:
db_tools = [list_tables, describe_table, execute_query]

instruction = """You are a helpful chatbot that can interact with an SQL database for a computer
store. You will take the users questions and turn them into SQL queries using the tools
available. Once you have the information you need, you will answer the user's question using
the data returned. ALWAYS start by calling list_tables to discover the available tables.
ALWAYS call describe_table on the relevant table before writing any SQL query.
Never assume or guess table names or column names — always verify first."""

from langgraph.prebuilt import create_react_agent

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0)

agent = create_react_agent(
    model=model,
    tools=db_tools,
    prompt=instruction,
)

Perhatikan respons chatbot di bawah, LLM memanggil tool `list_table` → `describe_table` → `execute_query`
untuk menganalisis jawaban dan memberikan jawaban yang tepat.

In [ ]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "Siapa saja orang yang terdaftar di database ini?"}]}
)
print(response["messages"][-1].content)

In [ ]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "Barang apa yang dijual di toko ini dan berapa harganya?"}]}
)
print(response["messages"][-1].content)

# AI Agents with LangGraph Part 2

## Langkah 1: Define State dan Prompt

In [ ]:
system_instruction = """
You are a BaristaBot, an interactive cafe ordering system.
A human will talk to you about the available products you have and you will answer any questions about menu items
(and only about menu items - no off-topic discussion, but you can chat about the products and their history).
The customer will place an order for 1 or more items from the menu, which you will structure and send to the ordering system after confirming the order with the human.

Add items to the customer's order with add_to_order, and reset the order with clear_order.
To see the contents of the order so far, call get_order (this is shown to you, not the user)
Always confirm_order with the user (double-check) before calling place_order.
Calling confirm_order will display the order items to the user and returns their response to seeing the list.
Their response may contain modifications. Always verify and respond with drink and modifier names from the MENU before adding them to the order.
If you are unsure a drink or modifier matches those on the MENU, ask a question to clarify or redirect.
You only have the modifiers listed on the menu.
Once the customer has finished ordering items, Call confirm_order to ensure it is correct then make any necessary updates and then call place_order.
Once place_order has returned, thank the user and say goodbye!
""" 

## Langkah 2: Define Flow Chatbot dengan create_react_agent

In [ ]:
customer_order = []

def get_menu() -> str:
    """Provide the latest up-to-date menu."""
    return """
    MENU:
    Coffee Drinks:
    Espresso, Americano, Cold Brew

    Coffee Drinks with Milk:
    Latte, Cappuccino, Cortado, Macchiato, Mocha, Flat White

    Tea Drinks:
    English Breakfast Tea, Green Tea, Earl Grey

    Tea Drinks with Milk:
    Chai Latte, Matcha Latte, London Fog

    Other Drinks:
    Steamer, Hot Chocolate

    Modifiers:
    Milk options: Whole, 2%, Oat, Almond, 2% Lactose Free; Default option: whole
    Espresso shots: Single, Double, Triple, Quadruple; default: Double
    Caffeine: Decaf, Regular; default: Regular
    Hot-Iced: Hot, Iced; Default: Hot
    Sweeteners (option to add one or more): vanilla sweetener, hazelnut sweetener, caramel sauce, chocolate sauce, sugar free vanilla sweetener

    "dirty" means add a shot of espresso to a drink that doesn't usually have it.
    "Regular milk" is the same as whole milk.
    Soy milk has run out of stock today, so soy is not available.
    """

def add_to_order(item: str) -> str:
    """Add an item to the customer's order."""
    global customer_order
    customer_order.append(item)
    print(f"Adding '{item}' to order. Current order: {customer_order}")
    return f"I've added '{item}' to your order."

def clear_order() -> str:
    """Clear all items from the customer's order."""
    global customer_order
    customer_order.clear()
    print("Clearing order.")
    return "Your order has been cleared."

def get_order() -> list[str]:
    """Get the current items in the customer's order."""
    global customer_order
    print(f"Getting order. Current order: {customer_order}")
    return customer_order

def confirm_order() -> str:
    """Confirm the order with the customer."""
    global customer_order
    if not customer_order:
        return "Your order is currently empty. What can I get for you?"
    return f"Your order contains: {', '.join(customer_order)}. Is this correct?"

def place_order() -> str:
    """Place the final order."""
    global customer_order
    if not customer_order:
        return "There's nothing in your order to place."
    summary = ", ".join(customer_order)
    customer_order.clear()
    return f"Your order for '{summary}' has been placed! It will be ready shortly."

barista_tools = [get_menu, add_to_order, clear_order, get_order, confirm_order, place_order]

from langgraph.checkpoint.memory import InMemorySaver

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0.2)
checkpointer = InMemorySaver()

barista_agent = create_react_agent(
    model=model,
    tools=barista_tools,
    prompt=system_instruction,
    checkpointer=checkpointer,
)

### Interaksi dengan Barista Bot

In [ ]:
def chat_with_barista():
    thread_id = "1"
    config = {"configurable": {"thread_id": thread_id}}
    print("Welcome to BaristaBot! Type 'q' to quit.")

    while True:
        user_input = input("You: ")
        if user_input.lower() in ['q', 'quit', 'exit']:
            print("BaristaBot: Thank you for visiting! Have a great day!")
            break

        response = barista_agent.invoke(
            {"messages": [{"role": "user", "content": user_input}]},
            config=config,
        )
        print("BaristaBot:", response["messages"][-1].content)

In [ ]:
# chat_with_barista()  # Uncomment to start interactive session

### Contoh Penggunaan Langsung

In [ ]:
thread_id = "3"
config = {"configurable": {"thread_id": thread_id}}

response = barista_agent.invoke(
    {"messages": [{"role": "user", "content": "What drinks do you have?"}]},
    config=config,
)
print("User: What drinks do you have?")
print("BaristaBot:", response["messages"][-1].content)

In [ ]:
response = barista_agent.invoke(
    {"messages": [{"role": "user", "content": "I'd like a green tea please"}]},
    config=config,
)
print("User: I'd like a green tea please")
print("BaristaBot:", response["messages"][-1].content)

In [ ]:
response = barista_agent.invoke(
    {"messages": [{"role": "user", "content": "what is my order?"}]},
    config=config,
)
print("User: what is my order?")
print("BaristaBot:", response["messages"][-1].content)